In [9]:
import pandas as pd

In [10]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier



In [11]:
df = pd.read_csv("../data/raw/accidents.csv")

In [12]:
df.head()

,grav,sexe,trajet,secu1,senc,obsm,choc,manv,motor,lum,...,nuit_sans_eclairage,surface_dangereuse,vitesse_x_collision,age_x_securite,agglo_x_vitesse,saison,est_mois_vacances,deux_roues,poids_lourd,vehicule_leger
0,3,1.0,1.0,1.0,1.0,2.0,1.0,1.0,5.0,2.0,...,0,0,80.0,0.0,80.0,3.0,0,0,0,0
1,1,1.0,1.0,1.0,1.0,9.0,3.0,17.0,1.0,2.0,...,0,0,80.0,43.0,80.0,3.0,0,0,0,1
2,4,1.0,5.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,...,0,0,0.0,38.0,80.0,3.0,0,0,0,1
3,3,1.0,5.0,1.0,2.0,2.0,1.0,9.0,1.0,1.0,...,0,0,0.0,28.0,80.0,3.0,0,0,0,1
4,1,1.0,1.0,1.0,1.0,1.0,3.0,1.0,1.0,1.0,...,0,0,0.0,26.0,100.0,2.0,1,0,0,1


In [13]:
X = df.drop(columns = "grav")
y = df["grav"]
y = y - 1
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 42)

In [14]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(405173, 33) (101294, 33) (405173,) (101294,)


In [15]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import mlflow.lightgbm

# 1. On pointe vers notre serveur local
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# 2. On crée (ou on réouvre) une expérience nommée
mlflow.set_experiment("accidents-gravite")

2026/02/25 12:59:17 INFO mlflow.tracking.fluent: Experiment with name 'accidents-gravite' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1772020757810, experiment_id='1', last_update_time=1772020757810, lifecycle_stage='active', name='accidents-gravite', tags={}, workspace='default'>

In [17]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score

# On ouvre un "run" = une session d'enregistrement
with mlflow.start_run(run_name="XGBoost-baseline"):
    
    # 1. Entraînement (ton code existant)
    model = XGBClassifier(max_depth=3, n_estimators=100, learning_rate=0.1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)
    
    # 2. Logger les hyperparamètres
    mlflow.log_params(model.get_params())
    
    # 3. Logger les métriques
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("f1", f1_score(y_test, y_pred, average="weighted"))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_proba,  average='weighted', multi_class='ovr'))
    mlflow.log_metric("precision", precision_score(y_test, y_pred, average="weighted"))
    mlflow.log_metric("recall", recall_score(y_test, y_pred, average="weighted"))
    
    # 4. Logger le modèle sérialisé
    mlflow.xgboost.log_model(model, artifact_path="model")

2026/02/25 13:02:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost-baseline at: http://127.0.0.1:5000/#/experiments/1/runs/1e0192ede6a5420396dde2a23e2eb61d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [18]:

models = {
    "LogisticRegression": (LogisticRegression(max_iter=1000), mlflow.sklearn),
    "XGBoost":            (XGBClassifier(n_estimators=100), mlflow.xgboost),
    "LightGBM":           (LGBMClassifier(n_estimators=100), mlflow.lightgbm),
    "RandomForest":       (RandomForestClassifier(n_estimators=20), mlflow.sklearn),
}

for run_name, (model, mlflow_module) in models.items():
    with mlflow.start_run(run_name=run_name):
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)
        
        mlflow.log_params(model.get_params())
        
        mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
        mlflow.log_metric("f1", f1_score(y_test, y_pred, average="weighted"))
        mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_proba,  average='weighted', multi_class='ovr'))
        mlflow.log_metric("precision", precision_score(y_test, y_pred, average="weighted"))
        mlflow.log_metric("recall", recall_score(y_test, y_pred, average="weighted"))
        
        mlflow_module.log_model(model, artifact_path="model")
        
        print(f"✅ {run_name} loggé")

c:\Users\ins expertise\anaconda3\envs\app\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/02/25 13:07:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/25 13:07:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe al

✅ LogisticRegression loggé
🏃 View run LogisticRegression at: http://127.0.0.1:5000/#/experiments/1/runs/628b67f62d444a22998474283c04b0d1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/02/25 13:08:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ XGBoost loggé
🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/1/runs/1e360c6fa05d41479b11471856fcb432
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.087143 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 462
[LightGBM] [Info] Number of data points in the train set: 405173, number of used features: 33
[LightGBM] [Info] Start training from score -0.856198
[LightGBM] [Info] Start training from score -3.619780
[LightGBM] [Info] Start training from score -1.886693
[LightGBM] [Info] Start training from score -0.924159


2026/02/25 13:09:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/25 13:09:04 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ LightGBM loggé
🏃 View run LightGBM at: http://127.0.0.1:5000/#/experiments/1/runs/b7db6b6c743b40b599326c3828572675
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/02/25 13:09:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/25 13:09:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ RandomForest loggé
🏃 View run RandomForest at: http://127.0.0.1:5000/#/experiments/1/runs/e6395a73edca47059f53de54597c1a8d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [19]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("accidents-gravite-tuning")

2026/02/25 13:18:30 INFO mlflow.tracking.fluent: Experiment with name 'accidents-gravite-tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1772021910126, experiment_id='2', last_update_time=1772021910126, lifecycle_stage='active', name='accidents-gravite-tuning', tags={}, workspace='default'>

In [20]:
configs = [
    {
        "max_depth": 3,
        "n_estimators": 100,
        "learning_rate": 0.1,
        "subsample": 1.0
    },
    {
        "max_depth": 5,
        "n_estimators": 200,
        "learning_rate": 0.05,
        "subsample": 0.8
    },
    {
        "max_depth": 7,
        "n_estimators": 300,
        "learning_rate": 0.01,
        "subsample": 0.8
    },
]

for i, params in enumerate(configs):
    with mlflow.start_run(run_name=f"XGBoost-config-{i+1}"):
        
        model = XGBClassifier(**params, random_state=42)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)
        
        # Logger les paramètres
        mlflow.log_params(params)
        
        # Logger les métriques
        mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
        mlflow.log_metric("f1", f1_score(y_test, y_pred, average='weighted'))
        mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted'))
        mlflow.log_metric("precision", precision_score(y_test, y_pred, average='weighted'))
        mlflow.log_metric("recall", recall_score(y_test, y_pred, average='weighted'))
        
        print(f"✅ Config {i+1} loggée")

✅ Config 1 loggée
🏃 View run XGBoost-config-1 at: http://127.0.0.1:5000/#/experiments/2/runs/7add2241e9dd42648360cc4e32cf9751
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
✅ Config 2 loggée
🏃 View run XGBoost-config-2 at: http://127.0.0.1:5000/#/experiments/2/runs/7f0d444e89264cca8033f6a8a87184ea
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
✅ Config 3 loggée
🏃 View run XGBoost-config-3 at: http://127.0.0.1:5000/#/experiments/2/runs/ad7b1db1987d43b4baf8c75c4aa4a704
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [21]:
# 📝 DOCUMENTATION — Meilleure configuration identifiée
# =====================================================
# Expérience  : accidents-gravite-tuning
# Modèle      : XGBoost
# 
# Config retenue :
#   - learning_rate : 0.05
#   - max_depth     : 5
#   - n_estimators  : 200
#   - subsample     : 0.8
#
# Métriques obtenues :
#   - Accuracy  : 0.652
#   - F1        : 0.631
#   - ROC AUC   : 0.827
#
# Justification :
#   Meilleur F1 et ROC AUC parmi les 3 configs testées.
#   Config 3 (max_depth=7) plus complexe mais moins performante
#   → signe d'overfitting. Config 2 retenue comme meilleur compromis.
#
# Note : le baseline (params défaut) reste légèrement meilleur en accuracy
#        (0.662 vs 0.652). Le tuning GridSearch/Optuna (Jour 3) permettra
#        d'explorer plus de combinaisons pour dépasser ce baseline.
# =====================================================

In [22]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # important : évite les problèmes d'affichage en dehors de jupyter

from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, confusion_matrix
from sklearn.preprocessing import label_binarize
import os

# Dossier temporaire pour sauvegarder les fichiers avant de les logger
os.makedirs("tmp_artifacts", exist_ok=True)

with mlflow.start_run(run_name="XGBoost-config2-avec-artefacts"):
    
    # 1. Entraînement avec la meilleure config
    params = {
        "max_depth": 5,
        "n_estimators": 200,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "random_state": 42
    }
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    # 2. Paramètres et métriques
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("f1", f1_score(y_test, y_pred, average='weighted'))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted'))

    # 3. Matrice de confusion
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
    ax.set_title("Matrice de confusion — XGBoost Config 2")
    fig.savefig("tmp_artifacts/confusion_matrix.png")
    plt.close(fig)
    mlflow.log_artifact("tmp_artifacts/confusion_matrix.png")

    # 4. Courbes ROC (une par classe)
    classes = sorted(y_test.unique())
    y_test_bin = label_binarize(y_test, classes=classes)

    fig, ax = plt.subplots(figsize=(8, 6))
    for i, classe in enumerate(classes):
        RocCurveDisplay.from_predictions(
            y_test_bin[:, i],
            y_proba[:, i],
            name=f"Classe {classe}",
            ax=ax
        )
    ax.set_title("Courbes ROC par classe — XGBoost Config 2")
    fig.savefig("tmp_artifacts/roc_curves.png")
    plt.close(fig)
    mlflow.log_artifact("tmp_artifacts/roc_curves.png")

    # 5. Feature names (métadonnées utiles)
    feature_names = X_train.columns.tolist()
    with open("tmp_artifacts/feature_names.txt", "w") as f:
        f.write("\n".join(feature_names))
    mlflow.log_artifact("tmp_artifacts/feature_names.txt")

    # 6. Modèle
    mlflow.xgboost.log_model(model, artifact_path="model")

    print("✅ Run avec artefacts loggé !")

2026/02/25 13:24:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ Run avec artefacts loggé !
🏃 View run XGBoost-config2-avec-artefacts at: http://127.0.0.1:5000/#/experiments/2/runs/5243076ddf8a4f16b41f31158a9d0608
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [23]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # important : évite les problèmes d'affichage en dehors de jupyter

from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, confusion_matrix
from sklearn.preprocessing import label_binarize
import os

# Dossier temporaire pour sauvegarder les fichiers avant de les logger
os.makedirs("tmp_artifacts", exist_ok=True)

with mlflow.start_run(run_name="XGBoost-config2-avec-artefacts"):
    
    # 1. Entraînement avec la meilleure config
    params = {
        "max_depth": 3,
        "n_estimators": 100,
        "learning_rate": 0.1,
        "subsample": 0.5,
        "random_state": 42
    }
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    # 2. Paramètres et métriques
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", accuracy_score(y_test, y_pred))
    mlflow.log_metric("f1", f1_score(y_test, y_pred, average='weighted'))
    mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted'))

    # 3. Matrice de confusion
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax)
    ax.set_title("Matrice de confusion — XGBoost Config 2")
    fig.savefig("tmp_artifacts/confusion_matrix.png")
    plt.close(fig)
    mlflow.log_artifact("tmp_artifacts/confusion_matrix.png")

    # 4. Courbes ROC (une par classe)
    classes = sorted(y_test.unique())
    y_test_bin = label_binarize(y_test, classes=classes)

    fig, ax = plt.subplots(figsize=(8, 6))
    for i, classe in enumerate(classes):
        RocCurveDisplay.from_predictions(
            y_test_bin[:, i],
            y_proba[:, i],
            name=f"Classe {classe}",
            ax=ax
        )
    ax.set_title("Courbes ROC par classe — XGBoost Config 2")
    fig.savefig("tmp_artifacts/roc_curves.png")
    plt.close(fig)
    mlflow.log_artifact("tmp_artifacts/roc_curves.png")

    # 5. Feature names (métadonnées utiles)
    feature_names = X_train.columns.tolist()
    with open("tmp_artifacts/feature_names.txt", "w") as f:
        f.write("\n".join(feature_names))
    mlflow.log_artifact("tmp_artifacts/feature_names.txt")

    # 6. Modèle
    mlflow.xgboost.log_model(model, artifact_path="model")

    print("✅ Run avec artefacts loggé !")

2026/02/25 13:29:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✅ Run avec artefacts loggé !
🏃 View run XGBoost-config2-avec-artefacts at: http://127.0.0.1:5000/#/experiments/2/runs/79f360661e8949479f421cf7a107326b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [24]:
from mlflow import MlflowClient

client = MlflowClient()

# Cherche dans TOUTES tes expériences le meilleur recall
best_run = mlflow.search_runs(
     experiment_names=[
        "accidents-gravite",
        "accidents-gravite-tuning"
    ],
    order_by=["metrics.recall DESC"]
).iloc[0]

print(f"✅ Meilleur run    : {best_run['tags.mlflow.runName']}")
print(f"   Recall         : {best_run['metrics.recall']}")
print(f"   ROC AUC        : {best_run['metrics.roc_auc']}")
print(f"   Run ID         : {best_run['run_id']}")

✅ Meilleur run    : XGBoost
   Recall         : 0.6621023950085888
   ROC AUC        : 0.8376957510322283
   Run ID         : 1e360c6fa05d41479b11471856fcb432


In [25]:
# 1. Construire l'URI du modèle à partir du run_id
model_uri = f"runs:/{best_run['run_id']}/model"

# 2. Enregistrer dans le Registry
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name="xgboost-gravite-accidents"
)

print(f"✅ Modèle enregistré !")
print(f"   Nom     : {registered_model.name}")
print(f"   Version : {registered_model.version}")

Successfully registered model 'xgboost-gravite-accidents'.
2026/02/25 13:32:14 WARNING mlflow.tracking._model_registry.fluent: Run with id 1e360c6fa05d41479b11471856fcb432 has no artifacts at artifact path 'model', registering model based on models:/m-de143550a73043b1bf9831d91f9dd6a1 instead
2026/02/25 13:32:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: xgboost-gravite-accidents, version 1
Created version '1' of model 'xgboost-gravite-accidents'.


✅ Modèle enregistré !
   Nom     : xgboost-gravite-accidents
   Version : 1


In [26]:
client = MlflowClient()

# Description du modèle
client.update_registered_model(
    name="xgboost-gravite-accidents",
    description="Modèle XGBoost de prédiction de gravité d'accidents routiers. "
                "Sélectionné sur le recall pour minimiser les faux négatifs sur cas graves."
)

# Description de cette version
client.update_model_version(
    name="xgboost-gravite-accidents",
    version=registered_model.version,
    description=f"Baseline XGBoost — recall={best_run['metrics.recall']:.3f} | "
                f"roc_auc={best_run['metrics.roc_auc']:.3f}"
)

print("✅ Description ajoutée !")

✅ Description ajoutée !


In [27]:
client.transition_model_version_stage(
    name="xgboost-gravite-accidents",
    version=registered_model.version,
    stage="Staging"
)

print(f"✅ Modèle v{registered_model.version} passé en Staging !")

✅ Modèle v1 passé en Staging !


C:\Users\ins expertise\AppData\Local\Temp\ipykernel_13740\763122162.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [28]:
# Charger depuis le Registry
model_loaded = mlflow.pyfunc.load_model(
    model_uri=f"models:/xgboost-gravite-accidents/Staging"
)

# Faire une prédiction de test
y_pred_test = model_loaded.predict(X_test[:5])
print(f"✅ Modèle rechargé depuis le Registry !")
print(f"   Prédictions test : {y_pred_test}")

✅ Modèle rechargé depuis le Registry !
   Prédictions test : [3 0 0 3 3]


In [33]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from mlflow import MlflowClient

mlflow.set_experiment("accidents-gravite-gridsearch")

# 1. Définir la grille
param_grid = {
    "max_depth":     [3, 5, 7],
    "n_estimators":  [100, 200, 300],
    "learning_rate": [0.01, 0.05, 0.1],
}

# 2. Lancer GridSearchCV
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=XGBClassifier(random_state=42, subsample=0.8),
    param_grid=param_grid,
    scoring="recall_weighted",   # notre métrique métier
    cv=cv,
    n_jobs=-1,                   # utilise tous les cœurs disponibles
    verbose=1
)

grid_search.fit(X_train, y_train)

# 3. Logger chaque combinaison comme un run MLflow distinct
for i, params in enumerate(grid_search.cv_results_["params"]):
    with mlflow.start_run(run_name=f"GridSearch-{i+1}"):

        # Réentraîner sur tout X_train avec ces params
        model = XGBClassifier(**params, subsample=0.8, random_state=42)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)

        # Logger
        mlflow.log_params(params)
        mlflow.log_metric("recall", recall_score(y_test, y_pred, average='weighted'))
        mlflow.log_metric("f1", f1_score(y_test, y_pred, average='weighted'))
        mlflow.log_metric("roc_auc", roc_auc_score(y_test, y_proba,
                                                     multi_class='ovr', average='weighted'))
        # Score CV (ce que GridSearch a vraiment optimisé)
        mlflow.log_metric("cv_recall", grid_search.cv_results_["mean_test_score"][i])

print(f"✅ {len(grid_search.cv_results_['params'])} runs loggés !")
print(f"   Meilleurs params : {grid_search.best_params_}")
print(f"   Meilleur CV recall : {grid_search.best_score_:.4f}")

2026/02/24 15:42:13 INFO mlflow.tracking.fluent: Experiment with name 'accidents-gravite-gridsearch' does not exist. Creating a new experiment.


Fitting 3 folds for each of 27 candidates, totalling 81 fits
🏃 View run GridSearch-1 at: http://127.0.0.1:5000/#/experiments/3/runs/10f7b2f0a3884af696ff39dc004a1485
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run GridSearch-2 at: http://127.0.0.1:5000/#/experiments/3/runs/eeb9f4ae72f04f4987330ff8fb886922
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run GridSearch-3 at: http://127.0.0.1:5000/#/experiments/3/runs/f287a24f9027457186acfad517058eed
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run GridSearch-4 at: http://127.0.0.1:5000/#/experiments/3/runs/21e7f9eb8b784669ab8e2149ce56c909
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run GridSearch-5 at: http://127.0.0.1:5000/#/experiments/3/runs/f242c8bfa39f45bdad8a4a8c5f9d55a8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run GridSearch-6 at: http://127.0.0.1:5000/#/experiments/3/runs/deb7cb6184af49798ef40658c7800384
🧪 View experime

In [ ]:
from optuna.integration.mlflow import MLflowCallback
import optuna

mlflowcb = MLflowCallback(
    tracking_uri="http://127.0.0.1:5000",
    metric_name="recall",
    create_experiment=False
)

C:\Users\ins expertise\AppData\Local\Temp\ipykernel_13740\2898039979.py:4: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflowcb = MLflowCallback(


In [30]:
@mlflowcb.track_in_mlflow()
def objective(trial):
    
    # Optuna choisit les paramètres
    params = {
        "max_depth":     trial.suggest_int("max_depth", 3, 10),
        "n_estimators":  trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample":     trial.suggest_float("subsample", 0.6, 1.0),
        "random_state":  42
    }
    
    # Même entraînement que d'habitude
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # On retourne la métrique qu'Optuna doit maximiser
    return recall_score(y_test, y_pred, average='weighted')

C:\Users\ins expertise\AppData\Local\Temp\ipykernel_13740\2466009481.py:1: ExperimentalWarning: optuna_integration.mlflow.mlflow.MLflowCallback.track_in_mlflow is experimental (supported from v2.9.0). The interface can change in the future.
  @mlflowcb.track_in_mlflow()


In [32]:
mlflow.set_experiment("accidents-gravite-optuna")

study = optuna.create_study(direction="maximize")  # on veut maximiser le recall
study.optimize(objective, n_trials=30, callbacks=[mlflowcb])

print(f"✅ Meilleur recall  : {study.best_value:.4f}")
print(f"   Meilleurs params : {study.best_params}")


2026/02/25 13:33:27 INFO mlflow.tracking.fluent: Experiment with name 'accidents-gravite-optuna' does not exist. Creating a new experiment.
[I 2026-02-25 13:33:27,416] A new study created in memory with name: no-name-70809e8f-76a0-44e2-a0e6-88e063e19d9c
2026/02/25 13:33:27 INFO mlflow.tracking.fluent: Experiment with name 'no-name-70809e8f-76a0-44e2-a0e6-88e063e19d9c' does not exist. Creating a new experiment.
[I 2026-02-25 13:36:15,119] Trial 0 finished with value: 0.6576401366319822 and parameters: {'max_depth': 9, 'n_estimators': 459, 'learning_rate': 0.2886173674544667, 'subsample': 0.7044953699756284}. Best is trial 0 with value: 0.6576401366319822.


🏃 View run 0 at: http://127.0.0.1:5000/#/experiments/4/runs/0b8b528e305049359db7ed4068ff38e9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
🏃 View run 0 at: http://127.0.0.1:5000/#/experiments/4/runs/0b8b528e305049359db7ed4068ff38e9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


[I 2026-02-25 13:38:48,480] Trial 1 finished with value: 0.6674432839062531 and parameters: {'max_depth': 9, 'n_estimators': 455, 'learning_rate': 0.05720608376237608, 'subsample': 0.9093071069446479}. Best is trial 1 with value: 0.6674432839062531.


🏃 View run 1 at: http://127.0.0.1:5000/#/experiments/4/runs/b7310d48bc8142629628824bf50752b7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
🏃 View run 1 at: http://127.0.0.1:5000/#/experiments/4/runs/b7310d48bc8142629628824bf50752b7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


[I 2026-02-25 13:40:30,731] Trial 2 finished with value: 0.6371848283215196 and parameters: {'max_depth': 4, 'n_estimators': 392, 'learning_rate': 0.016062824438453926, 'subsample': 0.9821477820506733}. Best is trial 1 with value: 0.6674432839062531.


...

🏃 View run 27 at: http://127.0.0.1:5000/#/experiments/4/runs/34cc7947465447e4a84f3846cc3dc8c9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
🏃 View run 27 at: http://127.0.0.1:5000/#/experiments/4/runs/34cc7947465447e4a84f3846cc3dc8c9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


[I 2026-02-25 14:27:07,192] Trial 28 finished with value: 0.664363140956029 and parameters: {'max_depth': 7, 'n_estimators': 306, 'learning_rate': 0.08380942416389649, 'subsample': 0.9650265282654371}. Best is trial 13 with value: 0.6679862578237605.


🏃 View run 28 at: http://127.0.0.1:5000/#/experiments/4/runs/61ff76ca53fe43608694c695ba86cb9b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
🏃 View run 28 at: http://127.0.0.1:5000/#/experiments/4/runs/61ff76ca53fe43608694c695ba86cb9b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


[I 2026-02-25 14:30:13,196] Trial 29 finished with value: 0.665903212431141 and parameters: {'max_depth': 9, 'n_estimators': 471, 'learning_rate': 0.023126821878461126, 'subsample': 0.6014892879084721}. Best is trial 13 with value: 0.6679862578237605.


🏃 View run 29 at: http://127.0.0.1:5000/#/experiments/4/runs/98d0702b70f14fc3b6168720087433ce
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
🏃 View run 29 at: http://127.0.0.1:5000/#/experiments/4/runs/98d0702b70f14fc3b6168720087433ce
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
✅ Meilleur recall  : 0.6680
   Meilleurs params : {'max_depth': 9, 'n_estimators': 495, 'learning_rate': 0.06870997241831617, 'subsample': 0.9164468824148215}
